In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import faiss

from sentence_transformers import SentenceTransformer

In [2]:
BASE_DIR = Path.cwd().parent

DATA_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "customer_support_tickets_processed_NLP.csv"
)

print(DATA_PATH)

/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /data/processed/customer_support_tickets_processed_NLP.csv


In [3]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

Shape: (8469, 21)


In [4]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,clean_text,text_length,word_count,clean_text_no_stopwords
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,i m having an issue with the product purchased...,265,45,issue product purchased please assist billing ...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,i m having an issue with the product purchased...,267,49,issue product purchased please assist need cha...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,i m facing a problem with my product purchased...,256,48,facing problem product purchased product purch...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,i m having an issue with the product purchased...,253,46,issue product purchased please assist problem ...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,i m having an issue with the product purchased...,321,59,issue product purchased please assist note sel...


In [5]:
support_text = df["clean_text_no_stopwords"].fillna("").astype(str)

In [6]:
print(support_text.head())

0    issue product purchased please assist billing ...
1    issue product purchased please assist need cha...
2    facing problem product purchased product purch...
3    issue product purchased please assist problem ...
4    issue product purchased please assist note sel...
Name: clean_text_no_stopwords, dtype: str


In [7]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
sample_text = support_text.iloc[0]

sample_embedding = model.encode(
    sample_text
)

In [9]:
sample_embedding

array([-3.94163057e-02,  4.52791490e-02,  2.66709421e-02, -3.75452153e-02,
       -3.65992337e-02, -1.26530463e-03, -2.86631174e-02, -3.19357887e-02,
       -1.41763799e-02,  2.69879159e-02, -4.54095379e-03,  9.23267379e-03,
        2.99497857e-03, -3.61951515e-02, -8.89523141e-03, -1.09019401e-02,
       -7.06700608e-02, -1.18980100e-02, -1.70832593e-02,  6.13922067e-02,
        6.31313492e-03,  1.23861767e-02, -5.01712933e-02,  2.28554271e-02,
        2.81661693e-02, -2.34278627e-02, -6.42543510e-02,  7.72743076e-02,
        6.13834709e-03, -3.31207849e-02,  5.88908233e-02,  4.52499688e-02,
        9.82298329e-02, -4.18363661e-02,  8.39014053e-02, -3.52072902e-02,
       -3.74423563e-02,  5.11048967e-03, -5.06901965e-02, -6.79278513e-03,
        1.21691357e-02, -6.96433112e-02, -3.09994426e-02,  3.55333462e-02,
        5.76164126e-02,  5.02696745e-02,  4.27736454e-02,  1.03466168e-01,
        8.34368318e-02,  4.64533977e-02, -4.77274507e-02, -8.49436969e-03,
        3.76063399e-03, -

In [10]:
print("Embedding type:", type(sample_embedding))
print("Embedding shape:", sample_embedding.shape)

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)


In [11]:
ticket_embeddings = model.encode(
    support_text.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/265 [00:00<?, ?it/s]

In [12]:
print("Number of tickets:", len(ticket_embeddings))
print("Embedding shape:", ticket_embeddings.shape)

Number of tickets: 8469
Embedding shape: (8469, 384)


In [13]:
print(
    "First embedding dimension:",
    len(ticket_embeddings[0])
)

First embedding dimension: 384


In [14]:
embedding_df = pd.DataFrame(
    ticket_embeddings
)

embedding_df.head()

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.039416,0.045279,0.026671,-0.037545,-0.036599,-0.001265,-0.028663,-0.031936,-0.014176,0.026988,...,-0.018723,-0.031233,-0.026983,-0.055430,0.013164,0.086104,0.001276,0.024054,-0.004747,-0.052615
1,-0.023082,-0.043112,0.050168,-0.057333,-0.019254,-0.000518,0.014286,-0.021230,0.009476,-0.013851,...,-0.004644,-0.015193,0.023840,-0.092151,0.019847,0.070310,0.026312,-0.016116,0.007568,0.008351
2,-0.047362,0.027635,0.022138,0.008822,-0.000689,-0.088814,-0.017343,-0.037595,-0.002791,-0.023878,...,0.019136,-0.000822,-0.053005,0.000117,0.005504,0.055913,0.052966,-0.057400,0.052815,0.076070
3,-0.035020,0.014701,0.005316,-0.037892,-0.032603,0.047978,-0.030006,-0.002476,-0.020400,-0.019661,...,-0.011064,0.002185,-0.005863,-0.056349,0.020848,0.064180,0.022417,0.004880,0.036391,0.053739
4,-0.043774,0.103077,0.041172,-0.076572,0.010585,0.033933,0.036723,0.060043,0.009493,0.103660,...,-0.017021,0.057163,0.027496,0.008777,0.075925,0.015810,0.001472,-0.063719,0.021072,0.079921


In [15]:
VECTOR_DIR = (
    BASE_DIR
    / "vector_store"
)

VECTOR_DIR.mkdir(
    exist_ok=True
)

In [16]:
np.save(
    VECTOR_DIR / "support_ticket_embeddings.npy",
    ticket_embeddings
)

In [17]:
print(
    "Saved:",
    VECTOR_DIR / "support_ticket_embeddings.npy"
)

Saved: /home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /vector_store/support_ticket_embeddings.npy


In [18]:
ticket_metadata = df[
    [
        "Ticket ID",
        "Ticket Type",
        "Ticket Subject",
        "Ticket Description"
    ]
].copy()

In [19]:
ticket_metadata

,Ticket ID,Ticket Type,Ticket Subject,Ticket Description
0,1,Technical issue,Product setup,I'm having an issue with the {product_purchase...
1,2,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...
2,3,Technical issue,Network problem,I'm facing a problem with my {product_purchase...
3,4,Billing inquiry,Account access,I'm having an issue with the {product_purchase...
4,5,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...
...,...,...,...,...
8464,8465,Product inquiry,Installation support,My {product_purchased} is making strange noise...
8465,8466,Technical issue,Refund request,I'm having an issue with the {product_purchase...
8466,8467,Technical issue,Account access,I'm having an issue with the {product_purchase...
8467,8468,Product inquiry,Payment issue,I'm having an issue with the {product_purchase...


In [20]:
ticket_metadata.to_csv(
    VECTOR_DIR / "support_ticket_metadata.csv",
    index=False
)

In [21]:
VECTOR_DIR = (
    BASE_DIR
    / "vector_store"
)

print(VECTOR_DIR)

/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /vector_store


In [22]:
ticket_embeddings.shape

(8469, 384)

In [23]:
print("Data type:", ticket_embeddings.dtype)

Data type: float32


In [24]:
ticket_metadata.head()

,Ticket ID,Ticket Type,Ticket Subject,Ticket Description
0,1,Technical issue,Product setup,I'm having an issue with the {product_purchase...
1,2,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...
2,3,Technical issue,Network problem,I'm facing a problem with my {product_purchase...
3,4,Billing inquiry,Account access,I'm having an issue with the {product_purchase...
4,5,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...


In [25]:
len(ticket_metadata)

8469

In [26]:
len(ticket_embeddings)

8469

In [27]:
embedding_dimension = 384

In [28]:
index = faiss.IndexFlatIP(
    embedding_dimension
)

In [29]:
index

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x739dbceb1330> >

In [30]:
index.ntotal

0

In [31]:
index.add(
    ticket_embeddings
)

In [32]:
index.ntotal

8469

In [33]:
index_path = (
    VECTOR_DIR
    / "support_ticket.index"
)

faiss.write_index(
    index,
    str(index_path)
)

print("FAISS index saved:")
print(index_path)

FAISS index saved:
/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /vector_store/support_ticket.index


In [34]:
query = "My internet connection is not working"

In [35]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

In [36]:
query_embedding

array([[ 6.86575286e-03, -9.51327849e-03,  6.07607327e-02,
        -7.97784925e-02, -1.09481893e-01, -3.56172919e-02,
         1.22414418e-02,  1.57965180e-02,  4.18075956e-02,
        -3.61729823e-02,  4.75159101e-02, -9.56286956e-03,
        -8.64146426e-02,  1.22057302e-02, -1.24676758e-02,
        -1.93581928e-03, -3.49871330e-02, -3.09691690e-02,
        -1.96774453e-02, -8.71364996e-02, -1.19357016e-02,
        -6.28391430e-02, -9.55812931e-02, -1.01328027e-02,
        -1.46650253e-02,  2.48572044e-02,  4.01480496e-02,
         3.66307795e-02, -3.64313200e-02, -3.51529755e-02,
        -3.05518527e-02, -6.57326803e-02,  8.32452625e-03,
        -6.05955012e-02,  1.22252933e-03,  1.79765280e-02,
        -3.02928835e-02,  6.86105937e-02,  1.49583295e-02,
        -1.56513918e-02, -2.25300025e-02, -5.98387532e-02,
         3.93384434e-02, -3.36294645e-03,  1.13341838e-01,
        -1.79956164e-02,  6.72911927e-02,  3.30456495e-02,
         9.02256444e-02,  1.52797494e-02, -9.61791072e-0

In [37]:
query_embedding.dtype

dtype('float32')

In [38]:
query_embedding.shape

(1, 384)

In [39]:
top_k = 5

similarity_scores, indices = index.search(
    query_embedding,
    top_k
)

In [40]:
print("Indices:")
print(indices)

print("\nSimilarity scores:")
print(similarity_scores)

Indices:
[[4427 6027 6259 5328 5520]]

Similarity scores:
[[0.60032284 0.59279406 0.5886868  0.58716804 0.5827979 ]]


In [41]:
results = ticket_metadata.iloc[
    indices[0]
].copy()

In [42]:
results

,Ticket ID,Ticket Type,Ticket Subject,Ticket Description
4427,4428,Refund request,Account access,The {product_purchased} is unable to establish...
6027,6028,Cancellation request,Network problem,The {product_purchased} is unable to establish...
6259,6260,Billing inquiry,Payment issue,The {product_purchased} is unable to establish...
5328,5329,Technical issue,Software bug,The {product_purchased} is unable to establish...
5520,5521,Billing inquiry,Battery life,The {product_purchased} is unable to establish...


In [43]:
results.insert(
    0,
    "Rank",
    range(1, top_k + 1)
)

In [44]:
results

,Rank,Ticket ID,Ticket Type,Ticket Subject,Ticket Description
4427,1,4428,Refund request,Account access,The {product_purchased} is unable to establish...
6027,2,6028,Cancellation request,Network problem,The {product_purchased} is unable to establish...
6259,3,6260,Billing inquiry,Payment issue,The {product_purchased} is unable to establish...
5328,4,5329,Technical issue,Software bug,The {product_purchased} is unable to establish...
5520,5,5521,Billing inquiry,Battery life,The {product_purchased} is unable to establish...


In [45]:
results["Similarity"] = similarity_scores[0]

In [46]:
results

,Rank,Ticket ID,Ticket Type,Ticket Subject,Ticket Description,Similarity
4427,1,4428,Refund request,Account access,The {product_purchased} is unable to establish...,0.600323
6027,2,6028,Cancellation request,Network problem,The {product_purchased} is unable to establish...,0.592794
6259,3,6260,Billing inquiry,Payment issue,The {product_purchased} is unable to establish...,0.588687
5328,4,5329,Technical issue,Software bug,The {product_purchased} is unable to establish...,0.587168
5520,5,5521,Billing inquiry,Battery life,The {product_purchased} is unable to establish...,0.582798


In [47]:
results[
    [
        "Rank",
        "Ticket ID",
        "Ticket Type",
        "Ticket Subject",
        "Similarity"
    ]
]

,Rank,Ticket ID,Ticket Type,Ticket Subject,Similarity
4427,1,4428,Refund request,Account access,0.600323
6027,2,6028,Cancellation request,Network problem,0.592794
6259,3,6260,Billing inquiry,Payment issue,0.588687
5328,4,5329,Technical issue,Software bug,0.587168
5520,5,5521,Billing inquiry,Battery life,0.582798


In [48]:
knowledge_documents = [

    # ============================================================
    # BILLING & PAYMENTS
    # ============================================================

    {
        "document_id": "KB001",
        "category": "Billing",
        "title": "Payment Failure",
        "content": """
If a payment fails, the customer should first verify that the payment
method has sufficient balance and that the card or payment account is
active. The customer can retry the payment after checking the payment
details. If the payment continues to fail, the customer should contact
customer support with the transaction details.
"""
    },

    {
        "document_id": "KB002",
        "category": "Billing",
        "title": "Refund Request",
        "content": """
Customers can request a refund by contacting customer support and
providing their order or transaction details. The support team should
verify the transaction before processing the request. Refund processing
time may depend on the payment method and transaction status.
"""
    },

    {
        "document_id": "KB003",
        "category": "Billing",
        "title": "Invoice Request",
        "content": """
Customers who need an invoice should contact customer support with
their order or transaction information. The support team can verify
the purchase and provide the appropriate invoice information.
"""
    },

    {
        "document_id": "KB013",
        "category": "Billing",
        "title": "Card Declined",
        "content": """
If a customer's card is declined, the customer should verify the card
number, expiration date, security information, and available balance.
The customer may retry the transaction after correcting any incorrect
details. If the card continues to be declined, the customer should
contact the card issuer or customer support.
"""
    },

    {
        "document_id": "KB014",
        "category": "Billing",
        "title": "Insufficient Balance",
        "content": """
A payment may fail when the selected payment account does not have
sufficient funds. Customers should verify their available balance
before retrying the transaction. The payment should only be retried
after sufficient funds are available.
"""
    },

    {
        "document_id": "KB015",
        "category": "Billing",
        "title": "Payment Pending",
        "content": """
When a payment remains in pending status, customers should avoid
making repeated payments for the same order. The payment status should
first be checked through the available transaction information.
If the payment remains pending for an extended period, support can
review the transaction status.
"""
    },

    {
        "document_id": "KB016",
        "category": "Billing",
        "title": "Duplicate Payment",
        "content": """
If a customer believes that the same order was charged more than once,
the customer should provide the order number and transaction details.
Support should compare the transactions and verify whether multiple
successful charges actually occurred before initiating a refund.
"""
    },

    {
        "document_id": "KB017",
        "category": "Billing",
        "title": "Payment Reversed",
        "content": """
A payment may be reversed when a previously processed transaction is
later cancelled or rejected by the payment system. Customers should
check their transaction history before attempting another payment.
If the order still shows as unpaid, support should review the payment
status.
"""
    },

    {
        "document_id": "KB018",
        "category": "Billing",
        "title": "Incorrect Billing Amount",
        "content": """
If a customer believes that the billed amount is incorrect, support
should verify the order price, applicable discounts, taxes, shipping
charges, and other fees. Customers should provide the order number and
billing information so that the charge can be investigated.
"""
    },

    {
        "document_id": "KB019",
        "category": "Billing",
        "title": "Unexpected Billing Charge",
        "content": """
Customers who notice an unexpected charge should provide the
transaction reference and account information to support. The support
team should verify the transaction and determine whether the charge
belongs to the customer's account or requires further investigation.
"""
    },

    {
        "document_id": "KB020",
        "category": "Billing",
        "title": "Payment Receipt",
        "content": """
Customers who require confirmation of a successful payment can
request a payment receipt. Support should verify the transaction before
providing the receipt or payment confirmation information.
"""
    },

    {
        "document_id": "KB021",
        "category": "Billing",
        "title": "Billing Address Change",
        "content": """
Customers who need to update their billing address should use the
available account settings when possible. If the billing address
cannot be changed through the account, support should verify the
customer information before making the required update.
"""
    },

    {
        "document_id": "KB022",
        "category": "Billing",
        "title": "Invoice Not Received",
        "content": """
If a customer has completed a purchase but has not received an
invoice, support should verify the transaction and registered contact
information. The invoice can be resent when the purchase has been
successfully verified.
"""
    },

    {
        "document_id": "KB023",
        "category": "Billing",
        "title": "Invoice Correction",
        "content": """
If an invoice contains incorrect information, the customer should
provide the invoice number and explain the required correction.
Support should verify the original transaction before issuing or
requesting a corrected invoice.
"""
    },

    {
        "document_id": "KB024",
        "category": "Billing",
        "title": "Tax Information",
        "content": """
Taxes may be included in the final transaction amount depending on the
customer's location and applicable billing rules. Customers who have
questions about tax information should provide the relevant invoice or
order details for review.
"""
    },

    {
        "document_id": "KB025",
        "category": "Billing",
        "title": "Transaction Verification",
        "content": """
Before making changes to billing information, refunds, or payment
records, support should verify the transaction using the available
order and customer details. Sensitive payment information should not be
requested unless it is required by an approved support procedure.
"""
    },


    # ============================================================
    # ACCOUNT & ACCESS
    # ============================================================

    {
        "document_id": "KB004",
        "category": "Account",
        "title": "Password Reset",
        "content": """
Customers who cannot access their account should use the password reset
option on the account login page. They should follow the instructions
sent to their registered email address. If the reset process does not
work, the customer should contact customer support.
"""
    },

    {
        "document_id": "KB005",
        "category": "Account",
        "title": "Account Access Problem",
        "content": """
If a customer cannot access their account, they should verify their
login credentials and check whether the account email is correct.
Customers should use the password recovery process when necessary.
Persistent access problems should be escalated to customer support.
"""
    },

    {
        "document_id": "KB026",
        "category": "Account",
        "title": "Forgotten Username",
        "content": """
Customers who have forgotten their username should use the account
recovery option if available. They may need to provide their registered
email address or other account information to verify ownership.
"""
    },

    {
        "document_id": "KB027",
        "category": "Account",
        "title": "Email Address Change",
        "content": """
Customers who want to change their registered email address should use
the account settings when the option is available. Account ownership
may need to be verified before the email address is changed.
"""
    },

    {
        "document_id": "KB028",
        "category": "Account",
        "title": "Account Verification",
        "content": """
Account verification may be required before support can make sensitive
changes to an account. Customers should provide only the information
requested through the approved verification process.
"""
    },

    {
        "document_id": "KB029",
        "category": "Account",
        "title": "Locked Account",
        "content": """
An account may become temporarily locked after repeated unsuccessful
login attempts or security-related activity. Customers should follow
the account recovery instructions. If access remains blocked, support
should verify the account and assist with recovery.
"""
    },

    {
        "document_id": "KB030",
        "category": "Account",
        "title": "Multiple Login Attempts",
        "content": """
Repeated unsuccessful login attempts can temporarily restrict account
access. Customers should stop repeated login attempts and use the
password recovery process instead of continuing to enter uncertain
credentials.
"""
    },

    {
        "document_id": "KB031",
        "category": "Account",
        "title": "Profile Information Update",
        "content": """
Customers can update permitted profile information through account
settings. Changes to sensitive account information may require
additional verification before they can be completed.
"""
    },

    {
        "document_id": "KB032",
        "category": "Account",
        "title": "Account Deactivation",
        "content": """
Customers who want to deactivate their account should review any
active subscriptions, pending orders, or unresolved transactions
before requesting deactivation. Support should verify the account
before processing the request.
"""
    },

    {
        "document_id": "KB033",
        "category": "Account",
        "title": "Account Reactivation",
        "content": """
A previously deactivated account may be eligible for reactivation
depending on the account status and applicable policy. Customers should
contact support with their registered account information for
verification.
"""
    },

    {
        "document_id": "KB034",
        "category": "Account",
        "title": "Suspicious Account Activity",
        "content": """
Customers who notice suspicious activity should secure their account
by changing the password and reviewing recent account activity. They
should contact support immediately if unauthorized activity is
suspected.
"""
    },

    {
        "document_id": "KB035",
        "category": "Account",
        "title": "Unauthorized Account Access",
        "content": """
If a customer believes that someone accessed the account without
permission, support should treat the issue as a security-related
request. The customer should secure the account and provide relevant
activity details for investigation.
"""
    },


    # ============================================================
    # CANCELLATION & REFUNDS
    # ============================================================

    {
        "document_id": "KB036",
        "category": "Cancellation",
        "title": "Order Cancellation",
        "content": """
Customers who want to cancel an order should contact customer support
as soon as possible and provide the order details. Cancellation
eligibility depends on the current order status and processing stage.
"""
    },

    {
        "document_id": "KB037",
        "category": "Cancellation",
        "title": "Cancellation Before Processing",
        "content": """
Orders that have not entered the processing stage may generally be
easier to cancel. Customers should submit cancellation requests as
early as possible and provide the order number for verification.
"""
    },

    {
        "document_id": "KB038",
        "category": "Cancellation",
        "title": "Cancellation After Processing",
        "content": """
Once an order has entered processing, cancellation may no longer be
available. Support should verify the current order status before
confirming whether cancellation can be completed.
"""
    },

    {
        "document_id": "KB039",
        "category": "Refund",
        "title": "Refund Eligibility",
        "content": """
Refund eligibility depends on the purchase type, order status, and
applicable refund policy. Support should verify the transaction and
review the relevant policy before confirming refund eligibility.
"""
    },

    {
        "document_id": "KB040",
        "category": "Refund",
        "title": "Refund Processing",
        "content": """
After a refund is approved, processing time may depend on the original
payment method and financial institution. Customers should retain the
refund reference information until the refund is completed.
"""
    },

    {
        "document_id": "KB041",
        "category": "Refund",
        "title": "Refund Not Received",
        "content": """
If an approved refund has not appeared in the customer's account,
support should verify the refund status and reference number. The
customer may also need to check with the original payment provider.
"""
    },

    {
        "document_id": "KB042",
        "category": "Refund",
        "title": "Partial Refund",
        "content": """
A partial refund may be appropriate when only part of an order or
service qualifies for reimbursement. Support should verify the
eligible amount and explain the refund calculation to the customer.
"""
    },

    {
        "document_id": "KB043",
        "category": "Refund",
        "title": "Refund Method",
        "content": """
Refunds are generally processed through the original payment method
when supported. Customers should provide the original transaction
details so that support can verify the correct refund destination.
"""
    },

    {
        "document_id": "KB044",
        "category": "Cancellation",
        "title": "Cancellation Confirmation",
        "content": """
After a cancellation is successfully processed, the customer should
receive or be provided with confirmation details. Customers should
retain the cancellation reference until the transaction is fully
settled.
"""
    },

    {
        "document_id": "KB045",
        "category": "Cancellation",
        "title": "Cancellation Request Escalation",
        "content": """
If a cancellation request cannot be completed through the standard
process, support should review the current order state and escalate
the request to the appropriate team when necessary.
"""
    },


    # ============================================================
    # DELIVERY & ORDERS
    # ============================================================

    {
        "document_id": "KB008",
        "category": "Delivery",
        "title": "Delayed Delivery",
        "content": """
If an order has not arrived within the expected delivery period,
customers should check the available order tracking information.
If the tracking information does not resolve the issue, the customer
should contact customer support with the order details.
"""
    },

    {
        "document_id": "KB046",
        "category": "Delivery",
        "title": "Order Tracking",
        "content": """
Customers can use the available tracking information to monitor the
status and movement of an order. If tracking information has not been
updated for an extended period, support can review the order status.
"""
    },

    {
        "document_id": "KB047",
        "category": "Delivery",
        "title": "Wrong Delivery Address",
        "content": """
Customers should verify the delivery address before submitting an
order. If the address is incorrect after an order is placed, the
customer should contact support immediately. Address changes may depend
on whether the order has already been shipped.
"""
    },

    {
        "document_id": "KB048",
        "category": "Delivery",
        "title": "Failed Delivery Attempt",
        "content": """
If a delivery attempt fails, customers should review the tracking
information for the reason and available next steps. A new delivery
attempt may depend on the carrier and current shipment status.
"""
    },

    {
        "document_id": "KB049",
        "category": "Delivery",
        "title": "Missing Package",
        "content": """
Customers who have not received a package marked as delivered should
first check the delivery location and with other authorized recipients.
If the package cannot be located, support should review the delivery
records and escalate the issue when necessary.
"""
    },

    {
        "document_id": "KB050",
        "category": "Delivery",
        "title": "Damaged Package",
        "content": """
Customers who receive a damaged package should report the issue as
soon as possible and provide the order details. Photographs or other
evidence may be requested to help support investigate the delivery
condition.
"""
    },

    {
        "document_id": "KB051",
        "category": "Delivery",
        "title": "Wrong Item Delivered",
        "content": """
If a customer receives an item different from the one ordered, support
should verify the order details and delivered product information.
Replacement or return options depend on the applicable order policy.
"""
    },

    {
        "document_id": "KB052",
        "category": "Delivery",
        "title": "Missing Item From Order",
        "content": """
If an order contains fewer items than expected, customers should
provide the order number and identify the missing item. Support should
compare the order records with fulfillment information before deciding
the appropriate resolution.
"""
    },

    {
        "document_id": "KB053",
        "category": "Delivery",
        "title": "Delivery Time Estimate",
        "content": """
Delivery estimates are provided based on the order processing stage,
shipping method, destination, and carrier conditions. Estimated
delivery dates are not always guaranteed and may change due to
unexpected delivery conditions.
"""
    },

    {
        "document_id": "KB054",
        "category": "Delivery",
        "title": "Shipping Method Change",
        "content": """
Requests to change the shipping method should be submitted before the
order is processed or shipped. Support should verify the current order
status before confirming whether a shipping method change is possible.
"""
    },

    {
        "document_id": "KB055",
        "category": "Delivery",
        "title": "Order Status Information",
        "content": """
Customers can check the current status of an order through the
available order tracking or account interface. Common statuses may
include processing, shipped, out for delivery, delivered, or
cancelled.
"""
    },


    # ============================================================
    # PRODUCT SUPPORT
    # ============================================================

    {
        "document_id": "KB006",
        "category": "Product Support",
        "title": "Product Not Working",
        "content": """
If a purchased product is not working, the customer should first check
the product setup and follow the basic troubleshooting instructions.
If the problem continues, the customer should contact support with the
product and purchase information.
"""
    },

    {
        "document_id": "KB007",
        "category": "Product Support",
        "title": "Product Replacement",
        "content": """
Customers requesting a product replacement should provide their order
details and explain the problem with the product. The support team
should verify the purchase and determine whether the replacement
request is eligible according to the applicable support policy.
"""
    },

    {
        "document_id": "KB056",
        "category": "Product Support",
        "title": "Product Setup Assistance",
        "content": """
Customers who need help setting up a product should follow the
official setup instructions and verify that all required components are
available. Support can provide additional guidance when the standard
setup process does not resolve the issue.
"""
    },

    {
        "document_id": "KB057",
        "category": "Product Support",
        "title": "Product Compatibility",
        "content": """
Before using a product with another device or service, customers
should verify compatibility requirements. If compatibility information
is unclear, support should review the product specifications and
customer use case.
"""
    },

    {
        "document_id": "KB058",
        "category": "Product Support",
        "title": "Product Defect Report",
        "content": """
Customers who believe a product has a manufacturing or functional
defect should provide the product details, purchase information, and a
description of the observed problem. Support should determine the
appropriate inspection, replacement, or repair process.
"""
    },

    {
        "document_id": "KB059",
        "category": "Product Support",
        "title": "Product Warranty",
        "content": """
Warranty support depends on the product, purchase date, warranty
conditions, and nature of the reported issue. Customers should provide
proof of purchase and product information when requesting warranty
assistance.
"""
    },

    {
        "document_id": "KB060",
        "category": "Product Support",
        "title": "Warranty Exclusion",
        "content": """
Warranty coverage may not apply to damage caused by misuse,
unauthorized modification, accidental damage, or conditions excluded
by the applicable warranty terms. Support should verify the warranty
conditions before confirming coverage.
"""
    },

    {
        "document_id": "KB061",
        "category": "Product Support",
        "title": "Replacement Eligibility",
        "content": """
Product replacement eligibility depends on the purchase record,
product condition, reported issue, and applicable replacement policy.
Support should verify these conditions before approving a replacement.
"""
    },

    {
        "document_id": "KB062",
        "category": "Product Support",
        "title": "Return Product for Inspection",
        "content": """
Some product issues may require the item to be returned for inspection.
Customers should follow the return instructions provided by support and
should not send products to an unapproved address.
"""
    },

    {
        "document_id": "KB063",
        "category": "Product Support",
        "title": "Product Maintenance",
        "content": """
Customers should follow the recommended maintenance and usage
instructions for their products. Proper maintenance can reduce
avoidable failures and help preserve product performance.
"""
    },

    {
        "document_id": "KB064",
        "category": "Product Support",
        "title": "Product Information Request",
        "content": """
Customers requesting product specifications should provide the product
name or model number. Support should provide information from approved
product documentation and should avoid making unsupported technical
claims.
"""
    },

    {
        "document_id": "KB065",
        "category": "Product Support",
        "title": "Product Availability",
        "content": """
Product availability may change based on inventory and fulfillment
conditions. Customers asking about availability should check the
current product information or contact support for the latest
availability details.
"""
    },


    # ============================================================
    # TECHNICAL SUPPORT
    # ============================================================

    {
        "document_id": "KB012",
        "category": "Technical Support",
        "title": "Technical Troubleshooting",
        "content": """
For technical problems, customers should first restart the affected
service or device and verify the basic configuration. Customers should
also check whether the issue continues after following the available
troubleshooting steps. If the problem remains unresolved, customer
support should be contacted.
"""
    },

    {
        "document_id": "KB066",
        "category": "Technical Support",
        "title": "Application Not Loading",
        "content": """
If an application does not load correctly, customers should check their
internet connection and restart the application. They should also
verify whether an updated version is available. Persistent problems
should be reported to technical support with device and error details.
"""
    },

    {
        "document_id": "KB067",
        "category": "Technical Support",
        "title": "Slow Application Performance",
        "content": """
Customers experiencing slow application performance should restart the
application and verify their network connection. They should provide
the affected feature, approximate time of occurrence, and device
information when reporting persistent performance issues.
"""
    },

    {
        "document_id": "KB068",
        "category": "Technical Support",
        "title": "Login Error",
        "content": """
Customers receiving a login error should verify their credentials and
confirm that the account is active. If the problem continues, they
should provide the exact error message and device information to
technical support.
"""
    },

    {
        "document_id": "KB069",
        "category": "Technical Support",
        "title": "Application Error Message",
        "content": """
When an application displays an error message, customers should record
the exact message and the action that caused it. Support can use this
information to identify the affected service and determine the
appropriate troubleshooting procedure.
"""
    },

    {
        "document_id": "KB070",
        "category": "Technical Support",
        "title": "Connectivity Problem",
        "content": """
For connectivity problems, customers should verify their internet
connection and retry the affected operation. If other services work
normally but the specific service remains unavailable, the issue
should be reported to technical support.
"""
    },

    {
        "document_id": "KB071",
        "category": "Technical Support",
        "title": "Service Outage",
        "content": """
If a service appears to be unavailable for multiple customers,
technical support should check the service status before asking
customers to repeat troubleshooting steps. Customers should be
informed when a known outage is identified.
"""
    },

    {
        "document_id": "KB072",
        "category": "Technical Support",
        "title": "Browser Troubleshooting",
        "content": """
For browser-related problems, customers can try refreshing the page,
clearing temporary browser data, disabling conflicting extensions, or
using a supported browser. Persistent issues should be reported with
browser and operating system information.
"""
    },

    {
        "document_id": "KB073",
        "category": "Technical Support",
        "title": "Mobile Application Issue",
        "content": """
Customers experiencing problems with a mobile application should
restart the application and verify that the latest supported version
is installed. If the issue continues, support should collect the
device model, operating system version, and error details.
"""
    },

    {
        "document_id": "KB074",
        "category": "Technical Support",
        "title": "Data Synchronization Problem",
        "content": """
If information does not synchronize correctly between supported
devices or services, customers should verify their connection and
refresh the affected application. Persistent synchronization problems
should be reported with details about the affected data.
"""
    },

    {
        "document_id": "KB075",
        "category": "Technical Support",
        "title": "Technical Issue Escalation",
        "content": """
Technical issues that cannot be resolved using standard troubleshooting
should be escalated to the appropriate technical team. The escalation
should include the error message, affected feature, troubleshooting
steps already attempted, and relevant technical information.
"""
    },


    # ============================================================
    # SUBSCRIPTIONS & PLANS
    # ============================================================

    {
        "document_id": "KB076",
        "category": "Subscription",
        "title": "Subscription Information",
        "content": """
Customers can review their current subscription plan through the
account interface when available. Subscription information may include
the plan type, billing frequency, renewal status, and applicable
features.
"""
    },

    {
        "document_id": "KB077",
        "category": "Subscription",
        "title": "Plan Upgrade",
        "content": """
Customers who want additional features or service capacity can request
a plan upgrade through the available account options. The customer
should review the updated pricing and plan conditions before confirming
the upgrade.
"""
    },

    {
        "document_id": "KB078",
        "category": "Subscription",
        "title": "Plan Downgrade",
        "content": """
Customers requesting a plan downgrade should review which features or
limits will change. The effective date of the downgrade may depend on
the subscription billing cycle and applicable plan rules.
"""
    },

    {
        "document_id": "KB079",
        "category": "Subscription",
        "title": "Subscription Renewal",
        "content": """
Subscriptions may renew according to the selected billing cycle.
Customers should review their renewal status and plan information
before the renewal date if they do not want the subscription to
continue.
"""
    },

    {
        "document_id": "KB080",
        "category": "Subscription",
        "title": "Subscription Cancellation",
        "content": """
Customers who want to cancel a subscription should use the available
cancellation option or contact support. The customer should review the
effective cancellation date and any remaining subscription benefits.
"""
    },

    {
        "document_id": "KB081",
        "category": "Subscription",
        "title": "Subscription Reactivation",
        "content": """
A cancelled subscription may be eligible for reactivation depending on
its current status and applicable plan rules. Support should verify the
subscription before confirming reactivation.
"""
    },

    {
        "document_id": "KB082",
        "category": "Subscription",
        "title": "Plan Feature Availability",
        "content": """
Different subscription plans may provide different features, limits,
or service levels. Customers should check the current plan information
before requesting a feature that may not be included in their plan.
"""
    },

    {
        "document_id": "KB083",
        "category": "Subscription",
        "title": "Subscription Billing Issue",
        "content": """
Customers who are incorrectly charged for a subscription should
provide the subscription information and transaction details. Support
should verify the active plan, billing cycle, and payment history
before resolving the issue.
"""
    },

    {
        "document_id": "KB084",
        "category": "Subscription",
        "title": "Expired Subscription",
        "content": """
When a subscription expires, access to subscription-specific features
may be restricted. Customers who want to continue using those features
should review the available renewal or reactivation options.
"""
    },

    {
        "document_id": "KB085",
        "category": "Subscription",
        "title": "Subscription Plan Comparison",
        "content": """
Customers comparing subscription plans should review pricing,
features, usage limits, billing frequency, and applicable conditions.
Support should provide information from the current approved plan
documentation.
"""
    },


    # ============================================================
    # CUSTOMER SUPPORT PROCEDURES
    # ============================================================

    {
        "document_id": "KB010",
        "category": "General Support",
        "title": "Support Escalation",
        "content": """
If a customer issue cannot be resolved through the standard support
process, the issue can be escalated to the appropriate support team.
The customer should provide the ticket number and relevant details so
that the issue can be reviewed efficiently.
"""
    },

    {
        "document_id": "KB011",
        "category": "General Support",
        "title": "Contact Customer Support",
        "content": """
Customers should provide their customer or order information when
contacting support. They should clearly describe the problem and
include relevant transaction or product details when applicable.
Providing complete information helps the support team investigate the
issue efficiently.
"""
    },

    {
        "document_id": "KB086",
        "category": "General Support",
        "title": "Ticket Information Requirements",
        "content": """
Support tickets should contain a clear description of the customer's
problem and relevant identifiers such as the order number, account
information, or transaction reference when applicable. Complete
information helps reduce unnecessary follow-up requests.
"""
    },

    {
        "document_id": "KB087",
        "category": "General Support",
        "title": "Ticket Priority",
        "content": """
Support ticket priority should be determined based on the impact,
urgency, number of affected customers, and business importance of the
issue. Critical service disruptions should receive higher priority
than routine informational requests.
"""
    },

    {
        "document_id": "KB088",
        "category": "General Support",
        "title": "Ticket Categorization",
        "content": """
Each support ticket should be assigned to the most appropriate
category based on the customer's primary issue. Accurate
categorization helps route the ticket to the correct support team and
improves reporting.
"""
    },

    {
        "document_id": "KB089",
        "category": "General Support",
        "title": "Ticket Assignment",
        "content": """
Tickets should be assigned to the team or agent responsible for the
reported issue. Tickets requiring specialized knowledge should be
transferred to the appropriate specialist rather than repeatedly
handled by unrelated teams.
"""
    },

    {
        "document_id": "KB090",
        "category": "General Support",
        "title": "Ticket Reopening",
        "content": """
A resolved ticket may be reopened when the original issue remains
unresolved or occurs again within the applicable support process.
Support should review the previous ticket history before taking new
action.
"""
    },

    {
        "document_id": "KB091",
        "category": "General Support",
        "title": "Duplicate Ticket Handling",
        "content": """
When multiple tickets describe the same customer issue, support should
identify the duplicate requests and consolidate the information when
appropriate. The customer should be informed which ticket will be used
for further communication.
"""
    },

    {
        "document_id": "KB092",
        "category": "General Support",
        "title": "Customer Information Verification",
        "content": """
Support agents should verify the customer's relevant account or order
information before making account-specific changes. Verification should
follow the approved support procedure and should not expose sensitive
information unnecessarily.
"""
    },

    {
        "document_id": "KB093",
        "category": "General Support",
        "title": "Escalation Information",
        "content": """
When escalating a ticket, the support agent should include the issue
summary, relevant customer information, troubleshooting already
performed, supporting evidence, and the reason for escalation. This
helps the receiving team investigate efficiently.
"""
    },


    # ============================================================
    # BUSINESS RULES & SERVICE POLICIES
    # ============================================================

    {
        "document_id": "KB094",
        "category": "Business Rules",
        "title": "Customer Data Privacy",
        "content": """
Customer information should be handled according to applicable
privacy and security requirements. Support agents should access only
the information necessary to resolve the customer's request and
should avoid sharing confidential information through unauthorized
channels.
"""
    },

    {
        "document_id": "KB095",
        "category": "Business Rules",
        "title": "Sensitive Information Handling",
        "content": """
Support agents should not request unnecessary passwords, authentication
secrets, or complete payment credentials from customers. Sensitive
information should be handled only through approved secure procedures.
"""
    },

    {
        "document_id": "KB096",
        "category": "Business Rules",
        "title": "Policy Verification",
        "content": """
Before approving exceptions, refunds, cancellations, replacements, or
other special requests, support should verify the applicable policy.
Agents should not promise an exception before the request has been
reviewed and approved.
"""
    },

    {
        "document_id": "KB097",
        "category": "Business Rules",
        "title": "Service Availability",
        "content": """
Service availability may depend on the customer's plan, account
status, product configuration, geographic availability, and current
system conditions. Support should verify the relevant service rules
before confirming availability.
"""
    },

    {
        "document_id": "KB098",
        "category": "Business Rules",
        "title": "Exception Approval",
        "content": """
Requests that fall outside standard policies should be submitted for
appropriate approval when an exception process exists. Support agents
should document the reason for the exception and the decision made by
the authorized reviewer.
"""
    },

    {
        "document_id": "KB099",
        "category": "Business Rules",
        "title": "Customer Communication Guidelines",
        "content": """
Support communication should be clear, professional, and focused on
the customer's issue. Agents should explain available next steps,
avoid unsupported promises, and provide accurate information based on
approved support procedures.
"""
    },

    {
        "document_id": "KB100",
        "category": "Business Rules",
        "title": "Issue Resolution Guidelines",
        "content": """
Support agents should first identify the customer's primary problem,
verify the relevant account or transaction information, and follow the
appropriate support procedure. If the issue cannot be resolved within
standard procedures, it should be escalated with complete supporting
information.
"""
    }

]

In [50]:
knowledge_documents

[{'document_id': 'KB001',
  'category': 'Billing',
  'title': 'Payment Failure',
  'content': '\nIf a payment fails, the customer should first verify that the payment\nmethod has sufficient balance and that the card or payment account is\nactive. The customer can retry the payment after checking the payment\ndetails. If the payment continues to fail, the customer should contact\ncustomer support with the transaction details.\n'},
 {'document_id': 'KB002',
  'category': 'Billing',
  'title': 'Refund Request',
  'content': '\nCustomers can request a refund by contacting customer support and\nproviding their order or transaction details. The support team should\nverify the transaction before processing the request. Refund processing\ntime may depend on the payment method and transaction status.\n'},
 {'document_id': 'KB003',
  'category': 'Billing',
  'title': 'Invoice Request',
  'content': '\nCustomers who need an invoice should contact customer support with\ntheir order or transaction 

In [51]:
kb_df = pd.DataFrame(knowledge_documents)

kb_df

,document_id,category,title,content
0,KB001,Billing,Payment Failure,"\nIf a payment fails, the customer should firs..."
1,KB002,Billing,Refund Request,\nCustomers can request a refund by contacting...
2,KB003,Billing,Invoice Request,\nCustomers who need an invoice should contact...
3,KB013,Billing,Card Declined,"\nIf a customer's card is declined, the custom..."
4,KB014,Billing,Insufficient Balance,\nA payment may fail when the selected payment...
...,...,...,...,...
94,KB096,Business Rules,Policy Verification,"\nBefore approving exceptions, refunds, cancel..."
95,KB097,Business Rules,Service Availability,\nService availability may depend on the custo...
96,KB098,Business Rules,Exception Approval,\nRequests that fall outside standard policies...
97,KB099,Business Rules,Customer Communication Guidelines,"\nSupport communication should be clear, profe..."


In [52]:
# clean knowledge base data

kb_df["content"] = (
    kb_df["content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [53]:
kb_df[["document_id", "title", "content"]]

,document_id,title,content
0,KB001,Payment Failure,"If a payment fails, the customer should first ..."
1,KB002,Refund Request,Customers can request a refund by contacting c...
2,KB003,Invoice Request,Customers who need an invoice should contact c...
3,KB013,Card Declined,"If a customer's card is declined, the customer..."
4,KB014,Insufficient Balance,A payment may fail when the selected payment a...
...,...,...,...
94,KB096,Policy Verification,"Before approving exceptions, refunds, cancella..."
95,KB097,Service Availability,Service availability may depend on the custome...
96,KB098,Exception Approval,Requests that fall outside standard policies s...
97,KB099,Customer Communication Guidelines,"Support communication should be clear, profess..."


In [54]:
# Create A Search Text

kb_df["text"] = (
    kb_df["title"] + ". " + kb_df["content"]
)

In [55]:
kb_df["text"]

0     Payment Failure. If a payment fails, the custo...
1     Refund Request. Customers can request a refund...
2     Invoice Request. Customers who need an invoice...
3     Card Declined. If a customer's card is decline...
4     Insufficient Balance. A payment may fail when ...
                            ...                        
94    Policy Verification. Before approving exceptio...
95    Service Availability. Service availability may...
96    Exception Approval. Requests that fall outside...
97    Customer Communication Guidelines. Support com...
98    Issue Resolution Guidelines. Support agents sh...
Name: text, Length: 99, dtype: str

In [56]:
kb_metadata = kb_df[
    [
        "document_id",
        "category",
        "title"
    ]
].copy()

kb_metadata

,document_id,category,title
0,KB001,Billing,Payment Failure
1,KB002,Billing,Refund Request
2,KB003,Billing,Invoice Request
3,KB013,Billing,Card Declined
4,KB014,Billing,Insufficient Balance
...,...,...,...
94,KB096,Business Rules,Policy Verification
95,KB097,Business Rules,Service Availability
96,KB098,Business Rules,Exception Approval
97,KB099,Business Rules,Customer Communication Guidelines


In [58]:
kb_df.to_csv(
   "../data/knowledge_base/knowledge_base.csv",
    index=False
)

print("Knowledge base saved.")

Knowledge base saved.


In [60]:
import json

with open(
    "../data/knowledge_base/knowledge_base.json",
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        knowledge_documents,
        f,
        indent=4,
        ensure_ascii=False
    )

print("JSON knowledge base saved.")

JSON knowledge base saved.
